# Lab 10: Machine translation

Guidelines: https://github.com/MohammedMosuily/NLP_lab10

Run cells from top to bottom from this folder.

## Task 1: Understanding the layers

1. **Embedding:** changes each word number into a vector that the model learns.
2. **Encoder LSTM:** reads the English sentence and makes a vector representing it.
3. **Decoder LSTM:** uses that vector to produce a sequence for the French sentence.
4. **Softmax:** gives a probability to each French word at each output position.
   The word with the highest probability is selected.

## Task 2: English–French translation

Dataset: https://www.kaggle.com/datasets/devicharith/language-translation-englishfrench

Use 10,000 short sentence pairs to keep training manageable. Keep one French
translation per English sentence, then split 80% for training and 20% for testing.
This is a small example of the method, not a reliable e-commerce translator.

In [1]:
from pathlib import Path
import re
import kagglehub
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split

tf.keras.utils.set_random_seed(42)
path = Path(kagglehub.dataset_download("devicharith/language-translation-englishfrench"))
df = pd.read_csv(path / "eng_-french.csv")
df.columns = ["english", "french"]
df = df.dropna()

def clean(text):
    return " ".join(re.sub(r"[^\w\s]", " ", text.lower()).split())

for column in df.columns:
    df[column] = df[column].apply(clean)

df = df[df["english"].str.split().str.len().between(1, 8)]
df = df[df["french"].str.split().str.len().between(1, 10)]
df = df.drop_duplicates("english")
# shortcut: use 10,000 short pairs for a local lab; use more data for better translations.
df = df.sample(n=10000, random_state=42)
train, test = train_test_split(df, test_size=0.2, random_state=42)
print("Training pairs:", len(train), "Testing pairs:", len(test))
print(train.head().to_string(index=False))

/Users/ali/Developer/nlp/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Training pairs: 8000 Testing pairs: 2000
                     english                              french
                    i rested                   je me suis reposé
                let me check                 laisse moi vérifier
 i ran into a little trouble je me suis heurté à quelques ennuis
i saw a town in the distance            je vis une ville au loin
   that almost made me laugh          cela m a presque fait rire


### Tokenize and pad

Learn word numbers from the training data only. Use `<unk>` for unseen words
and 0 for padding. English sequences have length 8 and French sequences length 10.

In [2]:
tokenizer_eng = Tokenizer(oov_token="<unk>")
tokenizer_frn = Tokenizer(oov_token="<unk>")
tokenizer_eng.fit_on_texts(train["english"])
tokenizer_frn.fit_on_texts(train["french"])

eng_train = pad_sequences(tokenizer_eng.texts_to_sequences(train["english"]), maxlen=8, padding="post")
eng_test = pad_sequences(tokenizer_eng.texts_to_sequences(test["english"]), maxlen=8, padding="post")
frn_train = pad_sequences(tokenizer_frn.texts_to_sequences(train["french"]), maxlen=10, padding="post")
frn_test = pad_sequences(tokenizer_frn.texts_to_sequences(test["french"]), maxlen=10, padding="post")

assert set(train["english"]).isdisjoint(test["english"])
assert eng_train.shape == (8000, 8) and frn_train.shape == (8000, 10)
print(eng_train.shape, frn_train.shape)

(8000, 8) (8000, 10)


### Build and train the LSTM

Use the same layers as the example: Embedding → LSTM → RepeatVector →
LSTM → Dense with softmax. The model learns to output 0 after the French words.
Test word accuracy ignores padding.

In [3]:
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(8,)),
    tf.keras.layers.Embedding(len(tokenizer_eng.word_index) + 1, 64, mask_zero=True),
    tf.keras.layers.LSTM(64),
    tf.keras.layers.RepeatVector(10),
    tf.keras.layers.LSTM(64, return_sequences=True),
    tf.keras.layers.TimeDistributed(tf.keras.layers.Dense(
        len(tokenizer_frn.word_index) + 1, activation="softmax"
    )),
])
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    weighted_metrics=[tf.keras.metrics.SparseCategoricalAccuracy(name="word_accuracy")],
)
model.summary()
history = model.fit(
    eng_train, frn_train,
    epochs=50, batch_size=64, validation_split=0.1, verbose=2,
)

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 8, 64)          │       262,272 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 64)             │        33,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ repeat_vector (RepeatVector)    │ (None, 10, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 10, 64)         │        33,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed                │ (None, 10, 5494)       │       357,110 │
│ (TimeDistributed)               │                        │               │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 685,430 (2.61 MB)

 Trainable params: 685,430 (2.61 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/50


... (95 outputs left out)

Epoch 49/50


113/113 - 1s - 13ms/step - loss: 3.0042 - val_loss: 3.8591 - val_word_accuracy: 0.4577 - word_accuracy: 0.4795


Epoch 50/50


113/113 - 1s - 13ms/step - loss: 2.9850 - val_loss: 3.8592 - val_word_accuracy: 0.4631 - word_accuracy: 0.4816


### Test and translate

Word accuracy measures individual output words, not whether a whole translation
is correct. Compare the predictions with the French references too.

In [4]:
results = model.evaluate(
    eng_test, frn_test, sample_weight=(frn_test != 0).astype("float32"),
    verbose=0, return_dict=True,
)
print("Test loss:", results["loss"])
print("Test word accuracy:", results["word_accuracy"])
print("Unknown French test words:", int((frn_test == 1).sum()))
assert np.isfinite(results["loss"])
assert history.history["loss"][-1] < history.history["loss"][0]

def translate_sentence(sentence):
    sequence = tokenizer_eng.texts_to_sequences([clean(sentence)])
    if not sequence[0] or len(sequence[0]) > 8:
        raise ValueError("Enter a sentence with 1 to 8 words.")
    padded = pad_sequences(sequence, maxlen=8, padding="post")
    prediction = model.predict(padded, verbose=0).argmax(axis=-1)[0]
    return " ".join(tokenizer_frn.index_word[int(i)] for i in prediction if i != 0)

for row in test.head(5).itertuples(index=False):
    print("English:", row.english)
    print("Reference:", row.french)
    print("Predicted:", translate_sentence(row.english))
    print()

for sentence in ["I want to buy this.", "How much does it cost?", "This product is good."]:
    print(sentence, "->", translate_sentence(sentence))

Test loss: 3.8414063453674316
Test word accuracy: 0.140353724360466
Unknown French test words: 774
English: i worked this morning
Reference: j ai travaillé ce matin
Predicted: je ai le de

English: i called and pretended i was tom
Reference: j ai téléphoné en prétendant être tom
Predicted: je ai que que que que ce

English: i want to know what s going on
Reference: je veux savoir ce qui se passe
Predicted: je ai que pas de

English: i already gave you half
Reference: je t ai déjà donné la moitié
Predicted: je ai que de de de

English: i do that for the money
Reference: je fais ça pour l argent


Predicted: je ai que de un

I want to buy this. -> je suis
How much does it cost? -> le moi
This product is good. -> c est est


The translations are poor. Test word accuracy is about 14%. This small model
would need more data and training before using it for product listings or
customer reviews.